In [ ]:
# load trained model

from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer
from src.utils import MODEL_PATH

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_PATH)

In [ ]:
from datasets import Dataset
from src.utils import load_cleaned_csv

cleaned_df = load_cleaned_csv()

dataset = Dataset.from_pandas(cleaned_df)

dataset = dataset.train_test_split(
    test_size=0.2,
    seed=42   # MUST match training
)

eval_dataset = dataset["test"]

In [ ]:
from legalbert_case_classifier.notebooks.nb_utils import trainer, eval_tokenized
from sklearn.metrics import classification_report
import numpy as np

output = trainer.predict(eval_tokenized)

labels = output.label_ids

preds = np.argmax(output.predictions, axis=-1)


target_names=["civil", "criminal", "legal_fees"]

print(classification_report(labels, preds, target_names=target_names, digits=4))

In [ ]:
# confusion matrix
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix

cm = confusion_matrix(labels, preds)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=target_names
)

plt.figure(figsize=(6, 6))
disp.plot(cmap="Blues", values_format="d")
plt.title("Confusion Matrix - LegalBERT Classification")
plt.tight_layout()

plt.savefig("confusion_matrix.png", dpi=300)
plt.show()